# Interactive VaR and CVaR Simulator

Module: Market Risk

## Lesson summary

This simulator lets students change tail probability, shock frequency, and shock size to see how VaR and Expected Shortfall react. It reinforces the difference between a quantile threshold and the average loss beyond that threshold.

## Learning objectives

By the end of this simulator, students should be able to:

- explain VaR as a quantile of the loss distribution;
- explain CVaR or Expected Shortfall as a tail conditional average;
- compare historical, Gaussian, Cornish-Fisher, and volatility-weighted VaR;
- describe how jump risk changes tail metrics;
- recognize why model choice affects reported capital.

## Tail metric definitions

The simulator displays risk metrics as positive loss numbers. For portfolio return $r$ and loss $L=-r$:

$$
\operatorname{VaR}_{\alpha}=Q_{1-\alpha}(L),
$$

$$
\operatorname{CVaR}_{\alpha}=\operatorname{ES}_{\alpha}
=\mathbb{E}\left[L\mid L\geq \operatorname{VaR}_{\alpha}\right].
$$

Changing shock probability or shock size changes the empirical tail of $L$, which is why VaR and CVaR do not move identically.

## Setup

In [ ]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from ipywidgets import FloatSlider, IntSlider, interact

from src.market_risk import (
    cornish_fisher_var,
    expected_shortfall,
    gaussian_var,
    historical_var,
    volatility_weighted_historical_var,
)

RUN_INTERACTIVE_WIDGETS = os.getenv("RUN_INTERACTIVE_WIDGETS", "1") == "1"

## Simulation helper

In [ ]:
def simulate_tail_returns(periods=1000, volatility=0.012, shock_probability=0.03, shock_size=-0.05, seed=181):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2022-01-03", periods=periods)
    base_returns = rng.normal(0.00025, volatility, periods)
    shocks = rng.binomial(1, shock_probability, periods) * rng.normal(
        shock_size,
        abs(shock_size) * 0.25,
        periods,
    )
    return pd.Series(base_returns + shocks, index=dates, name="portfolio_return")

## Interactive simulator

Run this cell in JupyterLab with `uv run jupyter lab`.

In [ ]:
def plot_var_cvar_simulator(
    alpha=0.01,
    volatility=0.012,
    shock_probability=0.03,
    shock_size=-0.05,
    lambda_=0.94,
    periods=1000,
):
    returns = simulate_tail_returns(
        periods=periods,
        volatility=volatility,
        shock_probability=shock_probability,
        shock_size=shock_size,
        seed=181,
    )
    risk_metrics = pd.Series(
        {
            "historical_var": historical_var(returns, alpha=alpha),
            "gaussian_var": gaussian_var(returns, alpha=alpha),
            "cornish_fisher_var": cornish_fisher_var(returns, alpha=alpha, validate_moments=False),
            "volatility_weighted_var": volatility_weighted_historical_var(
                returns,
                alpha=alpha,
                lambda_=lambda_,
            ),
            "expected_shortfall": expected_shortfall(returns, alpha=alpha),
        },
        name="positive_daily_loss",
    )

    fig = go.Figure()
    fig.add_trace(go.Histogram(x=returns, nbinsx=70, name="returns"))
    for metric, value in risk_metrics.items():
        fig.add_vline(
            x=-value,
            line_dash="dash",
            annotation_text=metric,
            annotation_position="top left",
        )
    fig.update_layout(
        title="VaR and Expected Shortfall simulator",
        xaxis_title="Daily return",
        yaxis_title="Frequency",
        template="plotly_white",
        height=520,
    )
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
    display(risk_metrics.to_frame())


if RUN_INTERACTIVE_WIDGETS:
    interact(
        plot_var_cvar_simulator,
        alpha=FloatSlider(value=0.01, min=0.005, max=0.10, step=0.005, readout_format=".3f"),
        volatility=FloatSlider(value=0.012, min=0.004, max=0.040, step=0.002, readout_format=".3f"),
        shock_probability=FloatSlider(value=0.03, min=0.00, max=0.15, step=0.01, readout_format=".2f"),
        shock_size=FloatSlider(value=-0.05, min=-0.20, max=-0.01, step=0.01, readout_format=".2f"),
        lambda_=FloatSlider(value=0.94, min=0.80, max=0.99, step=0.01, readout_format=".2f"),
        periods=IntSlider(value=1000, min=500, max=2000, step=250),
    );
else:
    plot_var_cvar_simulator()

## Model limitations

- The simulator changes stylized distribution parameters, not a fully calibrated market model.
- Parametric VaR can understate losses when skewness, kurtosis, or dependence differs from the assumed form.
- CVaR estimates can be noisy because they rely on relatively few observations in the tail.